# 01 — Exploratory Data Analysis

Confirms the EDA findings used to drive model design: 203 rm_ids, intermittent deliveries, August dip, top-volume rm_ids dominate.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd().parent))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.data import load_or_build, build_profile


In [ ]:
ds = load_or_build()
print('daily:', ds.daily.shape, '| rm_ids:', ds.daily['rm_id'].nunique())
print('profile_2024:', ds.profile_2024.shape)
print('prediction_mapping:', ds.prediction_mapping.shape)
print('materials:', ds.materials.shape)


## Per-rm intermittency profile
Fraction of rm_ids that fall into each gating track based on 2024 behaviour.

In [ ]:
from src.gating import assign_tracks, summarise_tracks
rm_ids = sorted(ds.prediction_mapping['rm_id'].unique().tolist())
tracks = assign_tracks(ds.profile_2024, all_rm_ids=rm_ids)
print(summarise_tracks(tracks))


## Top-10 rm_ids by 2024 volume
These drive the leaderboard. rm_id 2130 is the elephant — get it right or the score is ruined.

In [ ]:
top = ds.profile_2024.sort_values('total_kg', ascending=False).head(10)
top[['rm_id','total_kg','n_active_months','linear_r2','had_h2_delivery']]


## Annual volume — confirm the August dip
2024 monthly aggregates, summed across all rm_ids.

In [ ]:
month_total = (ds.daily[ds.daily['date'].dt.year==2024]
  .groupby(ds.daily['date'].dt.month)['daily_kg'].sum())
fig, ax = plt.subplots(figsize=(8,3))
month_total.plot.bar(ax=ax)
ax.set_xlabel('month'); ax.set_ylabel('total kg in 2024'); ax.set_title('2024 total deliveries by month'); plt.tight_layout()


## Year-over-year cumulative for the dominant rm_id (2130)

In [ ]:
rm = 2130
df = ds.daily[ds.daily['rm_id']==rm].copy()
df['year'] = df['date'].dt.year
df['doy'] = df['date'].dt.dayofyear
df = df[df['year'].between(2020, 2024)]
df['cum'] = df.groupby('year')['daily_kg'].cumsum()
fig, ax = plt.subplots(figsize=(8,4))
for year, g in df.groupby('year'):
    ax.plot(g['doy'], g['cum'], label=str(year))
ax.set_xlabel('day of year'); ax.set_ylabel('cumulative kg'); ax.set_title(f'rm_id {rm} cumulative deliveries by year'); ax.legend(); plt.tight_layout()


## Conclusion
The forecast for any rm_id is roughly a linear cumulative curve with a slope shaped by recent volume. The asymmetric pinball loss at τ=0.2 means we systematically shrink that slope. Sparse rm_ids contribute zero — predicting any positive value risks the 4× over-prediction penalty.